In [42]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(44)
n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),
    "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS").strftime("%Y-%m-%d"),
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"], n),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (pd.to_datetime("2026-03-01") + pd.to_timedelta(rng.integers(0, 60, n), unit="D")).strftime("%Y-%m-%d"),
})

idx = rng.choice(n, 30, replace=False)
txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")

txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)
txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)
print("Datasets generated successfully!")

Datasets generated successfully!


In [43]:
%%writefile cleaning_pipeline.py
import pandas as pd



def clean_agent_data(txns, agents):
    """Clean the raw agent transaction extract and merge it with the agent register.

    Parameters
    ----------
    txns : pd.DataFrame  raw transactions (txn_id, agent_id, txn_type, amount, txn_date)
    agents : pd.DataFrame  agent register (agent_id, county, onboarded)

    Returns
    -------
    clean_df : pd.DataFrame  cleaned transactions with county, amount_imputed, is_outlier
    log : pd.DataFrame  one row per cleaning rule: rule, dimension, column, rows_affected, action
    """
    df = txns.copy()
    log = []

    def record(rule, dimension, column, rows, action):
        log.append({"rule": rule, "dimension": dimension, "column": column,
                    "rows_affected": int(rows), "action": action})

    # 1. Uniqueness: duplicate txn_id
    dup = df.duplicated(subset="txn_id", keep="first")
    record("duplicate txn_id", "Uniqueness", "txn_id", dup.sum(), "removed, kept first")
    df = df[~dup].copy()

    # 2. Consistency: txn_type -> exactly Deposit / Withdrawal
    mapped = df["txn_type"].str.strip().str.lower().map(
        {"deposit": "Deposit", "withdrawal": "Withdrawal"})
    record("non-standard txn_type", "Consistency", "txn_type",
           (df["txn_type"] != mapped).sum(), "stripped, case-normalised")
    assert mapped.notna().all(), "unexpected txn_type value"
    df["txn_type"] = mapped

    # 3. Validity/type: txn_date with mixed separators -> datetime
    raw_date = df["txn_date"].astype(str)
    record("slash-formatted dates", "Consistency", "txn_date",
           raw_date.str.contains("/").sum(), "replaced / with -, parsed as ISO")
    df["txn_date"] = pd.to_datetime(raw_date.str.replace("/", "-"),
                                    format="%Y-%m-%d", errors="coerce")

    # 4. Validity/type: amount -> numeric (blank and "N/A" become NaN)
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    missing = df["amount"].isna()
    record("missing/invalid amount", "Completeness", "amount", missing.sum(),
           "flagged, imputed (step 6)")

    # 5. Accuracy: IQR outlier flag (on valid amounts only)
    q1, q3 = df["amount"].quantile([0.25, 0.75])
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    df["is_outlier"] = (df["amount"] < lo) | (df["amount"] > hi)   # NaN -> False
    record("IQR outlier", "Accuracy", "amount", df["is_outlier"].sum(),
           "flagged, not removed")

    # 6. Completeness: impute with the median per txn_type (non-outliers), flag it
    med = df.loc[~df["is_outlier"]].groupby("txn_type")["amount"].median()
    df["amount_imputed"] = missing
    df["amount"] = df["amount"].fillna(df["txn_type"].map(med))

    # 7. Merge: LEFT join so unknown agents are kept and counted
    out = df.merge(agents[["agent_id", "county"]], on="agent_id",
                   how="left", validate="m:1", indicator=True)
    unknown = out["_merge"] == "left_only"
    record("unknown agent_id", "Validity", "agent_id", unknown.sum(),
           "kept, county set to 'Unknown'")
    out["county"] = out["county"].fillna("Unknown")
    return out.drop(columns="_merge"), pd.DataFrame(log)

Overwriting cleaning_pipeline.py


In [44]:
import pandas as pd
import numpy as np

# Create mock agent transactions data
txns_data = {
    'txn_id': ['T101', 'T102', 'T103', 'T104', 'T101', 'T106', 'T107', 'T108'],  # T101 is duplicated
    'agent_id': ['A01', 'A02', 'A01', 'A03', 'A01', 'A99', 'A02', 'A03'],       # A99 is an unknown agent
    'txn_type': ['deposit', 'Withdrawal', ' DEPOSIT ', 'withdrawal', 'deposit', 'Deposit', 'withdrawal', 'Deposit'],
    'amount': [150.0, 30000.0, None, 'invalid_amount', 150.0, 200.0, 50.0, 120.0],  # 30000.0 is an outlier, None/invalid are missing
    'txn_date': ['2023-10-01', '2023/10/02', '2023-10-03', '2023-10-04', '2023-10-01', '2023-10-05', '2023/10/06', '2023-10-07']
}
txns_df = pd.DataFrame(txns_data)
txns_df.to_csv('agent_transactions.csv', index=False)

# Create mock agent register data
agents_data = {
    'agent_id': ['A01', 'A02', 'A03'],
    'county': ['Nairobi', 'Kiambu', 'Mombasa'],
    'onboarded': ['2021-01-15', '2022-03-10', '2021-11-01']
}
agents_df = pd.DataFrame(agents_data)
agents_df.to_csv('agents.csv', index=False)

print("Mock datasets 'agent_transactions.csv' and 'agents.csv' successfully created!")

Mock datasets 'agent_transactions.csv' and 'agents.csv' successfully created!


In [45]:
import pandas as pd
from cleaning_pipeline import clean_agent_data

txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")

clean_df, log = clean_agent_data(txns, agents)
log

,rule,dimension,column,rows_affected,action
0,duplicate txn_id,Uniqueness,txn_id,1,"removed, kept first"
1,non-standard txn_type,Consistency,txn_type,4,"stripped, case-normalised"
2,slash-formatted dates,Consistency,txn_date,2,"replaced / with -, parsed as ISO"
3,missing/invalid amount,Completeness,amount,2,"flagged, imputed (step 6)"
4,IQR outlier,Accuracy,amount,1,"flagged, not removed"
5,unknown agent_id,Validity,agent_id,1,"kept, county set to 'Unknown'"


In [62]:
%%writefile generate_data.py
# ---------- Engagement 1: Duka Electronics (seed 2026) ----------
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
n = 1000
products = {"Phone": 15000, "Laptop": 65000, "Headphones": 2500,
            "Tablet": 28000, "Charger": 1200}

sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "order_date": pd.to_datetime("2026-01-01")
                  + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "branch": rng.choice(["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"], n,
                         p=[0.4, 0.2, 0.15, 0.15, 0.1]),
    "product": rng.choice(list(products), n),
    "quantity": rng.integers(1, 6, n),
    "channel": rng.choice(["In-store", "Online", "Phone order"], n,
                          p=[0.55, 0.35, 0.10]),
})
sales["unit_price"] = sales["product"].map(products)
sales.to_csv("duka_sales.csv", index=False)

# ---------- Engagement 2: Agent network (seed 44) ----------
import numpy as np, pandas as pd

rng = np.random.default_rng(44)
n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),
    "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS")
                   .strftime("%Y-%m-%d"),
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT ",
                            "Withdrawal", "deposit"], n),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (pd.to_datetime("2026-03-01")
                 + pd.to_timedelta(rng.integers(0, 60, n), unit="D"))
                .strftime("%Y-%m-%d"),
})

idx = rng.choice(n, 30, replace=False)
txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")

txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)
txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

Overwriting generate_data.py


In [63]:
%run generate_data.py

In [48]:
import pandas as pd
from cleaning_pipeline import clean_agent_data

txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")

print(txns.shape)                          # expect (612, 5)
print(txns["agent_id"].eq("AG099").sum())  # expect 8

(612, 5)
8


In [49]:
clean_df, log = clean_agent_data(txns, agents)
display(log)
print(len(txns), "->", len(clean_df))

,rule,dimension,column,rows_affected,action
0,duplicate txn_id,Uniqueness,txn_id,12,"removed, kept first"
1,non-standard txn_type,Consistency,txn_type,376,"stripped, case-normalised"
2,slash-formatted dates,Consistency,txn_date,5,"replaced / with -, parsed as ISO"
3,missing/invalid amount,Completeness,amount,20,"flagged, imputed (step 6)"
4,IQR outlier,Accuracy,amount,40,"flagged, not removed"
5,unknown agent_id,Validity,agent_id,7,"kept, county set to 'Unknown'"


612 -> 600


In [50]:
print(clean_df["txn_type"].unique())                  # only Deposit, Withdrawal
print(clean_df["txn_id"].duplicated().sum())          # 0
print(clean_df.dtypes[["amount", "txn_date"]])        # float64, datetime64
print(clean_df["amount_imputed"].sum())               # 20
print(clean_df["county"].value_counts())              # includes Unknown: 7

# the suspicious values
clean_df[clean_df["amount"] == 250000]
clean_df[clean_df["is_outlier"]]["amount"].describe()

['Deposit' 'Withdrawal']
0
amount             float64
txn_date    datetime64[ns]
dtype: object
20
county
Nairobi    180
Mombasa    133
Kisumu     111
Nakuru      99
Kiambu      70
Unknown      7
Name: count, dtype: int64


,amount
count,40.000000
mean,39128.975000
std,80839.128538
min,6158.000000
25%,6707.000000
50%,7954.000000
75%,10410.000000
max,250000.000000


In [51]:
check = pd.read_csv("outputs/agent_transactions_clean.csv")
print(check.shape)             # expect (600, 8)
print(check.columns.tolist())
check.head()


(600, 8)
['txn_id', 'agent_id', 'txn_type', 'amount', 'txn_date', 'is_outlier', 'amount_imputed', 'county']


,txn_id,agent_id,txn_type,amount,txn_date,is_outlier,amount_imputed,county
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa
2,TX00003,AG008,Deposit,9129.0,2026-04-29,True,False,Kisumu
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru


In [52]:
import os
# Ensure the outputs directory exists
os.makedirs("outputs", exist_ok=True)

# Save the cleaned agent data to the expected output path
clean_df.to_csv("outputs/agent_transactions_clean.csv", index=False)
print("Saved cleaned data to outputs/agent_transactions_clean.csv")

Saved cleaned data to outputs/agent_transactions_clean.csv


In [53]:
# Now load and verify the file structure and inspect the log
check = pd.read_csv("outputs/agent_transactions_clean.csv")
print(check.shape)             # expect (600, 8)
print(check.columns.tolist())
display(check.head())

(600, 8)
['txn_id', 'agent_id', 'txn_type', 'amount', 'txn_date', 'is_outlier', 'amount_imputed', 'county']


,txn_id,agent_id,txn_type,amount,txn_date,is_outlier,amount_imputed,county
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa
2,TX00003,AG008,Deposit,9129.0,2026-04-29,True,False,Kisumu
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru


In [54]:
big = clean_df[clean_df["amount"] == 250000]
display(big)
print(big["agent_id"].value_counts())
print("share of total value:", big["amount"].sum() / clean_df["amount"].sum())


,txn_id,agent_id,txn_type,amount,txn_date,is_outlier,amount_imputed,county
306,TX00307,AG012,Deposit,250000.0,2026-03-02,True,False,Kisumu
315,TX00316,AG014,Deposit,250000.0,2026-04-01,True,False,Nairobi
424,TX00425,AG005,Withdrawal,250000.0,2026-03-02,True,False,Mombasa
463,TX00464,AG009,Withdrawal,250000.0,2026-04-11,True,False,Nakuru
594,TX00595,AG016,Deposit,250000.0,2026-03-11,True,False,Kiambu


agent_id
AG012    1
AG014    1
AG005    1
AG009    1
AG016    1
Name: count, dtype: int64
share of total value: 0.4652478468329649


In [55]:
raw = pd.read_csv("agent_transactions.csv", dtype=str, keep_default_na=False)
agents_raw = pd.read_csv("agents.csv")

amt = pd.to_numeric(raw.drop_duplicates("txn_id")["amount"], errors="coerce")
q1, q3 = amt.quantile([0.25, 0.75])
iqr_flags = ((amt < q1 - 1.5 * (q3 - q1)) | (amt > q3 + 1.5 * (q3 - q1))).sum()

audit = pd.DataFrame([
    ("Duplicate transactions (same txn_id)", "Uniqueness", "txn_id", raw.duplicated("txn_id").sum()),
    ("Inconsistent txn_type spelling/case/spaces", "Consistency", "txn_type", (~raw["txn_type"].isin(["Deposit", "Withdrawal"])).sum()),
    ("Dates written with / instead of -", "Consistency", "txn_date", raw["txn_date"].str.contains("/").sum()),
    ("Blank amount", "Completeness", "amount", (raw["amount"] == "").sum()),
    ('Amount recorded as "N/A"', "Completeness", "amount", (raw["amount"] == "N/A").sum()),
    ("Amount of exactly 250000 (suspect)", "Accuracy", "amount", (raw["amount"] == "250000").sum()),
    ("IQR outliers (after de-duplication)", "Accuracy", "amount", iqr_flags),
    ("agent_id not in register (AG099)", "Validity", "agent_id", (~raw["agent_id"].isin(agents_raw["agent_id"])).sum()),
], columns=["issue", "dimension", "column", "rows_affected"])
audit

,issue,dimension,column,rows_affected
0,Duplicate transactions (same txn_id),Uniqueness,txn_id,12
1,Inconsistent txn_type spelling/case/spaces,Consistency,txn_type,384
2,Dates written with / instead of -,Consistency,txn_date,5
3,Blank amount,Completeness,amount,15
4,"Amount recorded as ""N/A""",Completeness,amount,7
5,Amount of exactly 250000 (suspect),Accuracy,amount,5
6,IQR outliers (after de-duplication),Accuracy,amount,40
7,agent_id not in register (AG099),Validity,agent_id,8


In [56]:
import os
os.makedirs("outputs", exist_ok=True)
clean_df.to_csv("outputs/agent_transactions_clean.csv", index=False)


In [57]:
deduped = raw.drop_duplicates("txn_id")
print("raw rows:           ", len(raw))
print("after de-duplication:", len(deduped))
print("after merge (clean): ", len(clean_df))
print("rows preserved by left join:", len(clean_df) == len(deduped))
print("transactions with unknown agents:", (clean_df["county"] == "Unknown").sum())
print("duplicate txn_id:", clean_df["txn_id"].duplicated().sum())

raw rows:            612
after de-duplication: 600
after merge (clean):  600
rows preserved by left join: True
transactions with unknown agents: 7
duplicate txn_id: 0


In [58]:
county_summary = (
    clean_df.groupby(["county", "txn_type"])
    .agg(total_value=("amount", "sum"),
         n_txns=("txn_id", "count"),
         avg_value=("amount", "mean"))
    .round(0)
    .reset_index()
)
display(county_summary)


,county,txn_type,total_value,n_txns,avg_value
0,Kiambu,Deposit,334737.0,43,7785.0
1,Kiambu,Withdrawal,76920.0,27,2849.0
2,Kisumu,Deposit,436242.0,74,5895.0
3,Kisumu,Withdrawal,76244.0,37,2061.0
4,Mombasa,Deposit,148748.0,72,2066.0
5,Mombasa,Withdrawal,368592.0,61,6042.0
6,Nairobi,Deposit,543352.0,118,4605.0
7,Nairobi,Withdrawal,161514.0,62,2605.0
8,Nakuru,Deposit,184771.0,60,3080.0
9,Nakuru,Withdrawal,328464.0,39,8422.0


In [59]:
no_suspect = clean_df[clean_df["amount"] != 250000]
(no_suspect.groupby("county")["amount"].sum()
 .to_frame("total_excluding_250k")
 .join(clean_df.groupby("county")["amount"].sum().rename("total_all")))

,total_excluding_250k,total_all
county,,
Kiambu,161657.0,411657.0
Kisumu,262486.0,512486.0
Mombasa,267340.0,517340.0
Nairobi,454866.0,704866.0
Nakuru,263235.0,513235.0
Unknown,27156.0,27156.0


In [60]:
week = clean_df["txn_date"].dt.to_period("W").dt.start_time
weekly = clean_df.pivot_table(index=week, columns="county",
                              values="amount", aggfunc="sum", fill_value=0)
weekly = weekly.rename_axis("week_starting").round(0)
display(weekly)

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown
week_starting,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,2684.0,0.0
2026-03-02,18317.0,287933.0,285610.0,68967.0,56574.0,0.0
2026-03-09,267641.0,23186.0,30418.0,52801.0,28730.0,0.0
2026-03-16,22054.0,27935.0,25085.0,43048.0,39795.0,1639.0
2026-03-23,11504.0,27077.0,17050.0,41314.0,18868.0,0.0
2026-03-30,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0
2026-04-06,15030.0,15341.0,47033.0,56727.0,276609.0,9824.0
2026-04-13,12265.0,29297.0,26730.0,56302.0,26093.0,0.0
2026-04-20,24507.0,37439.0,16687.0,64926.0,30862.0,1877.0


## Data Quality Report: Agent Transactions

**Summary.** The extract had 612 rows; 600 remain after cleaning, with no valid transactions lost.

**Completeness.** 22 amounts were missing in the raw file (15 blank, 7 "N/A"); 20 remain after de-duplication.
They were imputed with the median amount for their transaction type and flagged in `amount_imputed`.
Using the median is standard practice here because it is robust to extreme outliers (such as the suspect 250,000.00 entries) which would heavily distort the mean. Rather than deleting these records and losing valuable operational transaction counts, imputation preserves the sample size while maintaining a realistic transaction value specific to each transaction type (Deposit vs. Withdrawal).

**Uniqueness.** 12 duplicate transactions (same `txn_id`) were removed, keeping the first.

**Validity.** `amount` converted to numeric; `txn_date` to datetime.
7 transactions reference agent `AG099`, which isn't in the register; kept, county set to "Unknown".

**Consistency.** 384 raw rows had non-standard `txn_type` (case, trailing spaces), standardised to Deposit/Withdrawal.
5 dates used "/" instead of "-" and were standardised to ISO format.

**Accuracy.** IQR flagged 40 amounts. 35 are ordinary values in the long tail of the distribution;
5 are exactly 250,000, one per county, about 47% of total value. Flagged, not deleted.
[Recommendation: verify with the source system, report totals with and without.]

**Timeliness.** Transactions run from 1 March to 29 April 2026. A cross-reference check comparing each agent's earliest transaction date against their onboarding date from the agent register confirmed that 100% of registered agents were successfully onboarded prior to their first transaction, indicating solid chronological alignment.

**Privacy.** [see below]

## Privacy and Personal Data

The current extract contains no customer identifiers, but if it also held customer
phone numbers, several fields would count as personal data under the Kenya Data
Protection Act 2019. The phone number is a direct identifier. Any name or national ID
number would be too. On their own, `txn_id`, `amount`, `txn_date` and `txn_type` are
not identifying, but combined with a phone number they reveal when a person moved
money and how much, which is information about their financial behaviour. Even without
phone numbers, `agent_id` joined to county and onboarding date could identify an
individual agent who operates as a sole trader, so the agent register should be
treated as personal data as well.

I would protect this data in the following ways. (1) Data minimisation: exclude phone
numbers from the analysis extract entirely, since none of the questions in this
engagement need them. (2) Pseudonymisation: where a customer-level link is needed
(for example, to count repeat customers), replace each phone number with a salted
hash, keep the salt separate from the dataset, and store the mapping back to real
numbers only with the data owner. (3) Security: encrypt files at rest and in transit,
restrict access to named analysts, and keep an access log. (4) Purpose and retention:
use the data only for the agent-performance analysis it was collected for, and delete
or anonymise it once that purpose is complete. (5) Cross-border transfer: do not move
the data outside Kenya or into personal accounts and unmanaged tools without a lawful
basis and safeguards. (6) Governance: carry out a data protection impact assessment
before processing at scale, register as a data controller or processor with the Office
of the Data Protection Commissioner as required, and follow the Act's breach
notification process if the data is exposed. Aggregated outputs such as the county
summary and weekly pivot contain no individual-level data and are safe to share.

In [61]:
first_txn = clean_df.groupby("agent_id")["txn_date"].min().rename("first_txn")
reg = agents.assign(onboarded=pd.to_datetime(agents["onboarded"])).set_index("agent_id")
chk = reg.join(first_txn, how="inner")
print("agents who transacted before onboarding:", (chk["onboarded"] > chk["first_txn"]).sum())
print(clean_df["txn_date"].min(), clean_df["txn_date"].max())

agents who transacted before onboarding: 0
2026-03-01 00:00:00 2026-04-29 00:00:00
